In [ ]:
from enum import StrEnum


class RuntimeEnvironment(StrEnum):
    colab = "colab"
    local = "local"

In [ ]:
RUNTIME_ENVIRONMENT = RuntimeEnvironment.colab

In [ ]:
if RUNTIME_ENVIRONMENT == RuntimeEnvironment.colab:
    !pip install lightning pydantic pydantic-settings optuna optuna-integration
    from google.colab import drive

    # Mount google drive
    drive.mount('/content/drive', force_remount=True)

    import os

    # Change working directory to coding environment
    from google.colab import userdata

    os.chdir(userdata.get("colab_cwd"))

In [ ]:
from pathlib import Path

cwd = Path.cwd()
print(cwd)

In [ ]:
from ta_module.config import DotEnv, Config, load_config, load_dot_env

DOT_ENV: DotEnv = load_dot_env()
CONFIG: Config = load_config(DOT_ENV.config_file)

print(f"DOT_ENV:\n{DOT_ENV}")
print(f"\nCONFIG:\n{CONFIG}")

In [ ]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Available device: {DEVICE}")

# Analisis karakteristik data

## Load data dari penyimpanan

In [ ]:
MORTALITAS_CONFIG = CONFIG.data.mortalitas

YEAR_COL = MORTALITAS_CONFIG.year_col
AGE_COL = MORTALITAS_CONFIG.age_col
SEX_COL = MORTALITAS_CONFIG.sex_col
MORTALITY_COL = MORTALITAS_CONFIG.mortality_col

BI_RATE_CONFIG = CONFIG.data.bi_rate

In [ ]:
import pandas as pd

mortalitas_df = pd.read_csv(
    DOT_ENV.mortalitas_file, parse_dates=[YEAR_COL], date_format=MORTALITAS_CONFIG.date_format
)

populasi_df = pd.read_csv(DOT_ENV.populasi_file)

bi_rate_df = pd.read_csv(
    DOT_ENV.bi_rate_file, parse_dates=[BI_RATE_CONFIG.date_col], date_format=BI_RATE_CONFIG.date_format
)

In [ ]:
AGE_MIN: int = min(mortalitas_df[AGE_COL])
AGE_MAX: int = max(mortalitas_df[AGE_COL])

YEAR_MIN: int = min(mortalitas_df[YEAR_COL].dt.year)
YEAR_MAX: int = max(mortalitas_df[YEAR_COL].dt.year)

## Analisa statistika deskriptif

In [ ]:
mortalitas_df.head()

In [ ]:
mortalitas_df.info()

In [ ]:
mortalitas_statdesc_df = mortalitas_df.groupby([SEX_COL, AGE_COL])[MORTALITY_COL].aggregate(
    ["mean", "std", "min", "median", "max"]
)
mortalitas_statdesc_df.to_csv(DOT_ENV.results_dir / "mortalitas_statdesc.csv")

In [ ]:
from ta_module.utils import plot_mortalitas_statdesc

mortalitas_statdesc_df = pd.read_csv(DOT_ENV.results_dir / "mortalitas_statdesc.csv")
plot_mortalitas_statdesc(df=mortalitas_statdesc_df, plots_dir=DOT_ENV.plots_dir)

In [ ]:
populasi_df.head()

In [ ]:
populasi_df.info()

In [ ]:
bi_rate_df.head()

In [ ]:
bi_rate_df.info()

## Histogram mortalitas

In [ ]:
import torch
import numpy as np

# Male mortalitas
mortalitas_df_male = mortalitas_df[mortalitas_df[SEX_COL] == "Male"]
M_male = mortalitas_df_male.pivot(
    index=YEAR_COL,
    columns=AGE_COL,
    values=MORTALITY_COL
)
M_male = torch.from_numpy(M_male.to_numpy(copy=True, dtype=np.float32)).to(DEVICE)

# Female mortalitas
mortalitas_df_female = mortalitas_df[mortalitas_df[SEX_COL] == "Female"]
M_female = mortalitas_df_female.pivot(
    index=YEAR_COL,
    columns=AGE_COL,
    values=MORTALITY_COL
)
M_female = torch.from_numpy(M_female.to_numpy(copy=True, dtype=np.float32)).to(DEVICE)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].hist(M_male.cpu().numpy().reshape(-1), color="blue", histtype="step")
ax[0].set_title("Laki-laki")
ax[0].set_xlabel("Mortality Rate")
ax[0].set_ylabel("Count")

ax[1].hist(M_female.cpu().numpy().reshape(-1), color="red", histtype="step")
ax[1].set_title("Perempuan")
ax[1].set_xlabel("Mortality Rate")
ax[1].set_ylabel("Count")

fig.suptitle("Distribusi Data Mortalitas")
fig.savefig(DOT_ENV.plots_dir / "distribusi_data_mortalitas.png")
plt.show()

## Plot mortalitas satu usia untuk semua tahun

In [ ]:
import seaborn as sns

sns.set_theme(style="whitegrid")

In [ ]:
plots_dir = DOT_ENV.plots_dir

In [ ]:
from ta_module.utils import plot_usia_vs_tahun


def wrapper_plot_usia_vs_tahun(age_start: int, age_end: int):
    plot_usia_vs_tahun(
        mortalitas_df=mortalitas_df,
        age_col=AGE_COL,
        year_col=YEAR_COL,
        sex_col=SEX_COL,
        mortality_col=MORTALITY_COL,
        age_start=age_start,
        age_end=age_end,
        plots_dir=plots_dir
    )

In [ ]:
wrapper_plot_usia_vs_tahun(AGE_MIN, AGE_MAX)

## Plot mortalitas semua usia untuk satu tahun

In [ ]:
from ta_module.utils import plot_tahun_vs_usia


def wrapper_plot_tahun_vs_usia(year_start: str, year_end: str):
    plot_tahun_vs_usia(
        mortalitas_df=mortalitas_df,
        age_col=AGE_COL,
        year_col=YEAR_COL,
        sex_col=SEX_COL,
        mortality_col=MORTALITY_COL,
        year_start=year_start,
        year_end=year_end,
        plots_dir=plots_dir
    )

In [ ]:
wrapper_plot_tahun_vs_usia(str(YEAR_MIN), str(YEAR_MAX))

# Persiapan data mortalitas untuk pelatihan model

## Transformasi data

In [ ]:
from ta_module.utils import ScaledLogitTransform

# Dari persamaan asumsi UDD qx = 2mx / (2 + mx)
# 0 <= 1x <= 1
# Jadi constraint untuk mx adalah:
lower_bound = 0.0
upper_bound = 2.0
link_fn = ScaledLogitTransform(lb=lower_bound, ub=upper_bound)

transformed_M_male = link_fn(M_male)
transformed_M_female = link_fn(M_female)

In [ ]:
TRAINING_CONFIG = CONFIG.training

## Pembagian data

In [ ]:
from ta_module.data import get_train_val_test_split

train_split = CONFIG.split.train
val_split = CONFIG.split.validation
test_split = CONFIG.split.test

M_male_train, M_male_val, M_male_test = get_train_val_test_split(
    mortality_matrix=transformed_M_male,
    train_split=train_split,
    val_split=val_split,
    test_split=test_split
)

M_female_train, M_female_val, M_female_test = get_train_val_test_split(
    mortality_matrix=M_female,
    train_split=train_split,
    val_split=val_split,
    test_split=test_split
)

In [ ]:
lookback = CONFIG.dataset.lookback
horizon = CONFIG.dataset.horizon

# Tambahkan konteks dari data train ke data val dan test agar sesuai dengan kebutuhan lookback
M_male_val_extend = torch.cat([M_male_train[-lookback:, :], M_male_val], dim=0)
M_male_test_extend = torch.cat([M_male_val_extend[-lookback:, :], M_male_test], dim=0)
M_female_val_extend = torch.cat([M_female_train[-lookback:, :], M_female_val], dim=0)
M_female_test_extend = torch.cat([M_female_val_extend[-lookback:, :], M_female_test], dim=0)

In [ ]:
import torch
import numpy as np

from ta_module.data import NormalizedMortalityDataset

train_male_mean = M_male_train.mean(dim=0)
train_male_std = M_male_train.std(dim=0)

pd.DataFrame({"mean": train_male_mean.cpu().numpy(), "std": train_male_std.cpu().numpy()}).to_csv(
    DOT_ENV.results_dir / "train_male_mean_std.csv", sep=";", decimal=","
)

## Normalisasi data

### Laki-laki

In [ ]:
create_male_dataset_split = NormalizedMortalityDataset.factory(
    lookback=lookback,
    horizon=horizon,
    mean=train_male_mean,
    std=train_male_std,
)

train_male_dataset = create_male_dataset_split(M_male_train)
val_male_dataset = create_male_dataset_split(M_male_val_extend)
test_male_dataset = create_male_dataset_split(M_male_test_extend)

In [ ]:
import matplotlib.pyplot as plt
from ta_module.utils import normalize

male_train_normalized = normalize(M_male_train, train_male_mean, train_male_std).cpu().numpy()
male_val_normalized = normalize(M_male_val, train_male_mean, train_male_std).cpu().numpy()
male_test_normalized = normalize(M_male_test, train_male_mean, train_male_std).cpu().numpy()

fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].plot(male_train_normalized.mean(axis=0), color="blue", linestyle="-", label="train")
ax[0].plot(male_val_normalized.mean(axis=0), color="red", linestyle="--", label="val")
ax[0].plot(male_test_normalized.mean(axis=0), color="orange", linestyle="-.", label="test")
ax[0].set_title("Mean")
ax[0].legend()
ax[0].set_xlabel("Age")
ax[0].set_ylabel("Mean value")

ax[1].plot(male_train_normalized.std(axis=0), color="blue", linestyle="-", label="train")
ax[1].plot(male_val_normalized.std(axis=0), color="red", linestyle="--", label="val")
ax[1].plot(male_test_normalized.std(axis=0), color="orange", linestyle="-.", label="test")
ax[1].set_title("Std")
ax[1].legend()
ax[1].set_xlabel("Age")
ax[1].set_ylabel("Std value")

fig.suptitle("Mean dan Std Data Mortalitas Laki-laki setelah Pemrosesan")
fig.savefig(DOT_ENV.plots_dir / "mean_std_mortalitas_laki-laki_setelah_pemrosesan.png")
plt.show()

### Perempuan

In [ ]:
train_female_mean = M_female_train.mean(dim=0)
train_female_std = M_female_train.std(dim=0)

pd.DataFrame({"mean": train_female_mean.cpu().numpy(), "std": train_female_std.cpu().numpy()}).to_csv(
    DOT_ENV.results_dir / "train_female_mean_std.csv", sep=";", decimal=","
)

In [ ]:
create_female_dataset_split = NormalizedMortalityDataset.factory(
    lookback=lookback,
    horizon=horizon,
    mean=train_female_mean,
    std=train_female_std,
)

train_female_dataset = create_female_dataset_split(M_female_train)
val_female_dataset = create_female_dataset_split(M_female_val_extend)
test_female_dataset = create_female_dataset_split(M_female_test_extend)

In [ ]:
female_train_normalized = normalize(M_female_train, train_female_mean, train_female_std).cpu().numpy()
female_val_normalized = normalize(M_female_val, train_female_mean, train_female_std).cpu().numpy()
female_test_normalized = normalize(M_female_test, train_female_mean, train_female_std).cpu().numpy()

fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].plot(female_train_normalized.mean(axis=0), color="blue", linestyle="-", label="train")
ax[0].plot(female_val_normalized.mean(axis=0), color="red", linestyle="--", label="val")
ax[0].plot(female_test_normalized.mean(axis=0), color="orange", linestyle="-.", label="test")
ax[0].set_title("Mean")
ax[0].legend()
ax[0].set_xlabel("Age")
ax[0].set_ylabel("Mean value")

ax[1].plot(female_train_normalized.std(axis=0), color="blue", linestyle="-", label="train")
ax[1].plot(female_val_normalized.std(axis=0), color="red", linestyle="--", label="val")
ax[1].plot(female_test_normalized.std(axis=0), color="orange", linestyle="-.", label="test")
ax[1].set_title("Std")
ax[1].legend()
ax[1].set_xlabel("Age")
ax[1].set_ylabel("Std value")

fig.suptitle("Mean dan Std Data Mortalitas Perempuan setelah Pemrosesan")
fig.savefig(DOT_ENV.plots_dir / "mean_std_mortalitas_perempuan_setelah_pemrosesan.png")
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].hist(male_train_normalized.reshape(-1), color="blue", histtype="step", label="train")
ax[0].hist(male_val_normalized.reshape(-1), color="red", histtype="step", label="val")
ax[0].hist(male_test_normalized.reshape(-1), color="orange", histtype="step", label="test")
ax[0].legend()
ax[0].set_title("Laki-laki")
ax[0].set_xlabel("Normalized Mortality Rate")
ax[0].set_ylabel("Count")

ax[1].hist(female_train_normalized.reshape(-1), color="blue", histtype="step", label="train")
ax[1].hist(female_val_normalized.reshape(-1), color="red", histtype="step", label="val")
ax[1].hist(female_test_normalized.reshape(-1), color="orange", histtype="step", label="test")
ax[1].legend()
ax[1].set_title("Perempuan")
ax[1].set_xlabel("Normalized Mortality Rate")
ax[1].set_ylabel("Count")

fig.suptitle("Distribusi Data Mortalitas setelah Pemrosesan")
fig.savefig(DOT_ENV.plots_dir / "distribusi_data_mortalitas_setelah_pemrosesan.png")
plt.show()

## Penggabungan data mortalitas laki-laki dan perempuan untuk pelatihan

In [ ]:
from torch.utils.data import ConcatDataset

train_dataset = ConcatDataset([train_male_dataset, train_female_dataset])
val_dataset = ConcatDataset([val_male_dataset, val_female_dataset])

In [ ]:
from torch.utils.data import DataLoader

batch_size = TRAINING_CONFIG.batch_size

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=len(val_dataset), shuffle=False)

# Pelatihan model

## Definisi arsitektur model

In [ ]:
MODEL_CONFIG = CONFIG.model

### LCN

In [ ]:
from torch import nn
from ta_module.models import LocalGLMnet, LocallyConnected2D

ukuran_matriks_input = (lookback, AGE_MAX - AGE_MIN + 1)
lcn_activation_function = nn.Sigmoid()
LCN_CONFIG = MODEL_CONFIG.lcn

create_lcn_layer = LocallyConnected2D.factory(
    input_size=ukuran_matriks_input,
    activation_fn=lcn_activation_function,
    kernel_size=LCN_CONFIG.kernel_size,
    zero_padding=LCN_CONFIG.zero_padding,
    bias=LCN_CONFIG.bias,
    device=DEVICE
)

### LocalGLMnet

In [ ]:
LOCALGLMNET_CONFIG = MODEL_CONFIG.localglmnet
create_localglmnet_model = LocalGLMnet.factory(
    input_size=ukuran_matriks_input,
    bias=LOCALGLMNET_CONFIG.bias,
    device=DEVICE
)

## Definisi metode pelatihan model

In [ ]:
TRAINING_CONFIG = CONFIG.training

### Loss function dan metrik evaluasi

In [ ]:
import torch.nn.functional as F

loss_metric_fn = F.mse_loss
eval_metric_fn = F.l1_loss

### Optimizer dan lr scheduler

In [ ]:
from typing import Iterator
from torch.optim import NAdam

OPTIMIZER_CONFIG = TRAINING_CONFIG.optimizer


def create_optimizer(params: Iterator[nn.Parameter]):
    return NAdam(
        params=params,
        lr=OPTIMIZER_CONFIG.lr,
    )

In [ ]:
from torch.optim import Optimizer
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingWarmRestarts, SequentialLR

LR_SCHEDULER_CONFIG = TRAINING_CONFIG.lr_scheduler


def create_lr_scheduler(optimizer: Optimizer):
    warm_up_epochs = LR_SCHEDULER_CONFIG.total_iters
    warm_up_scheduler = LinearLR(
        optimizer=optimizer,
        start_factor=LR_SCHEDULER_CONFIG.start_factor,
        end_factor=LR_SCHEDULER_CONFIG.end_factor,
        total_iters=warm_up_epochs
    )

    sgdr_scheduler = CosineAnnealingWarmRestarts(
        optimizer=optimizer,
        T_0=LR_SCHEDULER_CONFIG.T_0,
        T_mult=LR_SCHEDULER_CONFIG.T_mult,
        eta_min=LR_SCHEDULER_CONFIG.eta_min
    )

    return SequentialLR(
        optimizer=optimizer,
        schedulers=[warm_up_scheduler, sgdr_scheduler],
        milestones=[warm_up_epochs]
    )

## Grid search untuk hyperparameter koefisien regularisasi LASSO

In [ ]:
TUNING_CONFIG = CONFIG.tuning

In [ ]:
from optuna.trial import Trial
from typing import Callable, Iterator
from torch.nn import Parameter

from ta_module.tuning import reg_coef_grid_search, reg_coef_objective
from ta_module.config import TuneMetadata, ModeEnum
from ta_module.utils import get_current_run_datetime, ElasticNetRegularizationTerm

if CONFIG.mode == ModeEnum.TUNE:
    max_epochs = TRAINING_CONFIG.max_epochs
    min_epochs = TRAINING_CONFIG.min_epochs
    reg_coef_candidates = TUNING_CONFIG.reg_coef_candidates


    def create_mymodel_with_reg_coef(
        reg_coef: float
    ):
        def _create_regularization_term(model_weights_getter: Callable[[], Iterator[Parameter]]):
            return ElasticNetRegularizationTerm(
                reg_coef=reg_coef,
                alpha=TRAINING_CONFIG.regularization.alpha,
                model_weights_getter=model_weights_getter,
            )

        localglmnet = create_localglmnet_model(create_lcn_layer())
        localglmnet_attention_weights_getter = lambda: (
            params for name, params in
            localglmnet.regression_attention_model.named_parameters()
            if "bias" not in name
        )

        return MyModel(
            model=localglmnet,
            loss_metric=loss_metric_fn,
            eval_metric=eval_metric_fn,
            create_optimizer=create_optimizer,
            create_lr_scheduler=create_lr_scheduler,
            regularization_term=_create_regularization_term(model_weights_getter=localglmnet_attention_weights_getter),
        )


    objective: Callable[[Trial], float] = lambda trial: (
        reg_coef_objective(
            trial=trial,
            create_my_model_with_reg_coef=create_mymodel_with_reg_coef,
            train_dataloader=train_dataloader,
            val_dataloader=val_dataloader,
            max_epochs=max_epochs,
            min_epochs=min_epochs,
            log_dir=DOT_ENV.tuning_logs_dir,
            checkpoint_dir=DOT_ENV.tuning_checkpoints_dir,
            reg_coef_candidates=reg_coef_candidates,
            gradient_clip_val=TRAINING_CONFIG.regularization.gradient_clip_val
        )
    )

    run_datetime = get_current_run_datetime()
    print(f"==================================================================")
    print("Mulai grid search untuk hyperparameter reg_coef dalam lasso loss")
    print(f"Run datetime: {run_datetime}")
    print(f"reg_coef candidates: {reg_coef_candidates}")
    print(f"==================================================================\n")
    grid_search_result = reg_coef_grid_search(
        objective_fn=objective,
        reg_coef_candidates=reg_coef_candidates,
        storage=DOT_ENV.optuna_db_url,
        seed=CONFIG.seed,
    )
    print(f"\n==================================================================\n")
    print("Grid_search selesai!")
    print(f"Result:\n{grid_search_result}\n")

    last_tune_metadata_filepath = DOT_ENV.last_tune_metadata_file
    print(f"Update file {last_tune_metadata_filepath}:")
    tune_metadata: TuneMetadata = TuneMetadata.model_validate(
        {
            "datetime": run_datetime,
            "result"  : grid_search_result,
        }
    )
    print(f"Tune metadata:\n{tune_metadata}\n")
    with open(last_tune_metadata_filepath, "w") as f:
        f.write(tune_metadata.model_dump_json(indent=4))
    print(f"Update berhasil!")
else:
    print(f"Mode = {CONFIG.mode}")
    print("Skip proses tuning")

## Pelatihan model dengan hyperparameter terbaik

### Load hyperparameter terbaik dari tune metadata

In [ ]:
from ta_module.config import load_last_tune_metadata

last_tune_metadata = load_last_tune_metadata(DOT_ENV.last_tune_metadata_file)
print(f"Last tune metadata:\n{last_tune_metadata}")

In [ ]:
tune_trials_result = pd.DataFrame(last_tune_metadata.result.trials)
tune_trials_result.to_csv(DOT_ENV.results_dir / "tune_trials_result.csv", sep=";", decimal=",")

In [ ]:
best_reg_coef = last_tune_metadata.result.best_params.get("reg_coef", None)
create_elasticnet_regularization = ElasticNetRegularizationTerm.factory(
    reg_coef=best_reg_coef,
    alpha=TRAINING_CONFIG.regularization.alpha,
)

### Proses pelatihan

In [ ]:
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import TensorBoardLogger, CSVLogger
from lightning import Trainer

from ta_module.config import TrainMetadata
from ta_module.models import MyModel
from ta_module.utils import get_current_run_datetime_str

if CONFIG.mode in [ModeEnum.TRAIN, ModeEnum.TUNE]:
    num_ensembles = MODEL_CONFIG.num_ensembles
    localglmnet_models = [
        create_localglmnet_model(create_lcn_layer())

        for _ in range(num_ensembles)
    ]

    localglmnet_attention_weight_getters = [
        lambda: (params for name, params in localglmnet.regression_attention_model.named_parameters() if
                 "bias" not in name)
        for localglmnet in localglmnet_models
    ]

    logalglmnet_mymodels = [
        MyModel(
            model=localglmnet_models[i],
            loss_metric=loss_metric_fn,
            eval_metric=eval_metric_fn,
            regularization_term=create_elasticnet_regularization(localglmnet_attention_weight_getters[i]),
            create_optimizer=create_optimizer,
            create_lr_scheduler=create_lr_scheduler
        )

        for i in range(num_ensembles)
    ]

    max_epochs = TRAINING_CONFIG.max_epochs
    min_epochs = TRAINING_CONFIG.min_epochs
    run_datetime = get_current_run_datetime()
    run_datetime_str = get_current_run_datetime_str()
    model_names = [f"LocalGLMnet_{i + 1}" for i in range(num_ensembles)]
    checkpoint_dirs = [DOT_ENV.training_checkpoints_dir / model_names[i] for i in range(num_ensembles)]

    trainers = [
        Trainer(
            max_epochs=max_epochs,
            min_epochs=min_epochs,
            log_every_n_steps=1,
            deterministic=True,
            gradient_clip_val=TRAINING_CONFIG.regularization.gradient_clip_val,
            logger=[
                TensorBoardLogger(
                    save_dir=DOT_ENV.training_logs_dir,
                    name=model_names[i],
                    version=run_datetime_str
                ),
                CSVLogger(
                    save_dir=DOT_ENV.training_logs_dir,
                    name=model_names[i],
                    version=run_datetime_str,
                )
            ],
            callbacks=[
                ModelCheckpoint(
                    dirpath=checkpoint_dirs[i],
                    filename=run_datetime_str,
                    monitor="val_loss",
                    mode="min",
                    save_top_k=1
                )
            ]
        )

        for i in range(num_ensembles)
    ]

    # Train ensembles pada data mortalitas laki-laki dan perempuan
    print(f"Melatih {num_ensembles} model LocalGLMnet secara independen")
    print(f"Run datetime: {run_datetime_str}")
    for i, (trainer, model) in enumerate(zip(trainers, logalglmnet_mymodels)):
        print("\n================================================================")
        print(f"Training model {i + 1} pada mortalitas laki-laki dan perempuan:")
        print("================================================================\n")
        trainer.fit(
            model=model,
            train_dataloaders=train_dataloader,
            val_dataloaders=val_dataloader,
        )

    print("Pelatihan selesai!")

    # Save metadadata
    last_train_metadata_filepath = DOT_ENV.last_train_metadata_file
    checkpoint_file_paths = [checkpoint_dirs[i] / f"{run_datetime_str}.ckpt" for i in range(num_ensembles)]
    print(f"Update file {last_train_metadata_filepath}:")
    # metadata digunakan untuk load checkpoint model terakhir secara otomatis
    # jika tidak run proses pelatihan
    train_metadata = TrainMetadata.model_validate(
        {
            "datetime"             : run_datetime,
            "checkpoint_file_paths": checkpoint_file_paths,
        }, extra="forbid"
    )
    print(f"Train metadata:\n{train_metadata}\n")
    with open(last_train_metadata_filepath, "w") as f:
        f.write(train_metadata.model_dump_json(indent=4))
    print("Update berhasil!")
else:
    print(f"Mode = {CONFIG.mode}")
    print("Skip proses pelatihan, langsung inference")

# Evaluasi model LocalGLMnet

In [ ]:
from ta_module.config import load_last_train_metadata

last_train_metadata = load_last_train_metadata(DOT_ENV.last_train_metadata_file)

In [ ]:
last_train_metadata

In [ ]:
localglmnet_models = []
for checkpoint_filepath in last_train_metadata.checkpoint_file_paths:
    localglmnet_model = create_localglmnet_model(create_lcn_layer())
    ckpt = torch.load(checkpoint_filepath, map_location=DEVICE)
    state_dict = ckpt["state_dict"]

    # sesuaikan dengan nama atribut di LightningModule-mu
    prefix = "model."
    state_dict = {
        k[len(prefix):]: v
        for k, v in state_dict.items()
        if k.startswith(prefix)
    }

    localglmnet_model.load_state_dict(state_dict)
    localglmnet_model.eval()
    localglmnet_models.append(localglmnet_model)

print(localglmnet_models)

In [ ]:
from ta_module.models import EnsembleLocalGLMNet

localglmnet_ensemble = EnsembleLocalGLMNet(
    models=localglmnet_models,
    device=DEVICE
)

print(localglmnet_ensemble)

In [ ]:
test_male_dataloader = DataLoader(test_male_dataset, batch_size=len(test_male_dataset), shuffle=False)
male_test_loss = 0.0
male_test_score = 0.0

for batch in test_male_dataloader:
    x, y = batch
    with torch.no_grad():
        y_pred = localglmnet_ensemble(x)

    y_detransformed = link_fn.inv(y)
    y_pred_detransformed = link_fn.inv(y_pred)

    male_test_loss = F.mse_loss(y_detransformed, y_pred_detransformed).detach()
    male_test_score = F.l1_loss(y_detransformed, y_pred_detransformed).detach()

print(f"Male test MSE = {male_test_loss:.6f}")
print(f"Male test MAE = {male_test_score:.6f}")

In [ ]:
test_female_dataloader = DataLoader(test_female_dataset, batch_size=len(test_male_dataset), shuffle=False)
female_test_loss = 0.0
female_test_score = 0.0

for batch in test_female_dataloader:
    x, y = batch
    with torch.no_grad():
        y_pred = localglmnet_ensemble(x)

    y_detransformed = link_fn.inv(y)
    y_pred_detransformed = link_fn.inv(y_pred)

    female_test_loss = F.mse_loss(y_detransformed, y_pred_detransformed).detach()
    female_test_score = F.l1_loss(y_detransformed, y_pred_detransformed).detach()

print(f"Female test MSE = {female_test_loss:.6f}")
print(f"Female test MAE = {female_test_score:.6f}")

In [ ]:
print(f"Average MSE = {(male_test_loss + female_test_loss) / 2:.6f}")
print(f"Average MAE = {(male_test_score + female_test_score) / 2:.6f}")

# Interpretasi pemodelan

In [ ]:
# TODO

# Simulasi peramalan mortalitas

## Laki-laki

In [ ]:
male_residuals = None

male_dataset = NormalizedMortalityDataset(
    mortality_matrix=transformed_M_male,
    lookback=lookback,
    horizon=horizon,
    mean=train_male_mean,
    std=train_male_std,
)

male_dataloader = DataLoader(male_dataset, batch_size=len(male_dataset), shuffle=False)
for batch in male_dataloader:
    x, y = batch
    with torch.no_grad():
        y_pred = localglmnet_ensemble(x)
    male_residuals = y - y_pred

print(male_residuals[0])
print(male_residuals.shape)

In [ ]:
male_residuals[0, 0, :]

In [ ]:
plt.plot(male_residuals[0, 0, :])
plt.show()

In [ ]:
from ta_module.utils import recursive_forecast_with_residual_bootstrap, denormalize
import torch

male_simulations_file_path = DOT_ENV.results_dir / "male_mortality_simulations.npy"
if not male_simulations_file_path.exists():
    x, y = male_dataset[-1]
    # Dimensi = (10, W)
    x_in = torch.cat([x[1:, :], y])

    # Ubah menjadi dimensi = (1, 10, W)
    x_in = x_in.unsqueeze(0)

    male_mortality_simulations = recursive_forecast_with_residual_bootstrap(
        model=localglmnet_ensemble,
        x=x_in,
        residuals=male_residuals,
        forecast_horizon=55,
        n_sim=10_000,
        normalize_mean=train_male_mean,
        normalize_std=train_male_std,
        device=DEVICE
    )
    # Denormalize
    male_mortality_simulations = denormalize(male_mortality_simulations, train_male_mean, train_male_std)
    # Transformasi rentang nilai: R -> (0, 2)
    male_mortality_simulations = link_fn.inv(male_mortality_simulations)
    np.save(male_simulations_file_path, male_mortality_simulations.cpu().numpy())
else:
    male_mortality_simulations = np.load(male_simulations_file_path)

print(male_mortality_simulations[0])
print(male_mortality_simulations.shape)

## Mortalitas perempuan

In [ ]:
female_residuals = None

female_dataset = NormalizedMortalityDataset(
    mortality_matrix=transformed_M_female,
    lookback=lookback,
    horizon=horizon,
    mean=train_male_mean,
    std=train_male_std,
)

female_dataloader = DataLoader(female_dataset, batch_size=len(male_dataset), shuffle=False)
for batch in female_dataloader:
    x, y = batch
    with torch.no_grad():
        y_pred = localglmnet_ensemble(x)
    female_residuals = y - y_pred

print(female_residuals[0])
print(female_residuals.shape)

In [ ]:
female_simulations_file_path = DOT_ENV.results_dir / "female_mortality_simulations.npy"
if not female_simulations_file_path.exists():
    x, y = female_dataset[-1]
    # Dimensi = (10, W)
    x_in = torch.cat([x[1:, :], y])

    # Ubah menjadi dimensi = (1, 10, W)
    x_in = x_in.unsqueeze(0)

    female_mortality_simulations = recursive_forecast_with_residual_bootstrap(
        model=localglmnet_ensemble,
        x=x_in,
        residuals=female_residuals,
        forecast_horizon=55,
        n_sim=10_000,
        normalize_mean=train_female_mean,
        normalize_std=train_female_std,
        device=DEVICE
    )

    # Denormalize
    female_mortality_simulations = denormalize(female_mortality_simulations, train_female_mean, train_female_std)
    # Transformasi rentang nilai: R -> (0, 2)
    female_mortality_simulations = link_fn.inv(female_mortality_simulations)
    np.save(female_simulations_file_path, female_mortality_simulations.cpu().numpy())
else:
    female_mortality_simulations = np.load(female_simulations_file_path)

print(female_mortality_simulations[0])
print(female_mortality_simulations.shape)

# Pembuatan life table

## Laki-laki

In [ ]:
male_mortality_simulations[0]

In [ ]:
male_1qx = (2.0 * male_mortality_simulations) / (2.0 + male_mortality_simulations)
male_1px = 1.0 - male_1qx

In [ ]:
male_1px[0]